In [ ]:
!pip install transformers sentencepiece keybert nltk plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import gzip
import json

from tqdm import tqdm

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving reviews_Clothing_Shoes_and_Jewelry_5.json.gz to reviews_Clothing_Shoes_and_Jewelry_5.json.gz


In [ ]:
import os

print(os.listdir('/content'))

['.config', 'reviews_Clothing_Shoes_and_Jewelry_5.json.gz', 'sample_data']


In [ ]:
data = []

with gzip.open('/content/reviews_Clothing_Shoes_and_Jewelry_5.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        data.append(json.loads(line))

df = pd.DataFrame(data)

278677it [00:03, 75722.72it/s]


In [ ]:
print(df.shape)
print(df.columns)
df.head()

(278677, 9)
Index(['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText',
       'overall', 'summary', 'unixReviewTime', 'reviewTime'],
      dtype='object')


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A1KLRMWW2FWPL4,0000031887,"Amazon Customer ""cameramom""","[0, 0]",This is a great tutu and at a really great pri...,5.0,Great tutu- not cheaply made,1297468800,"02 12, 2011"
1,A2G5TCU2WDFZ65,0000031887,Amazon Customer,"[0, 0]",I bought this for my 4 yr old daughter for dan...,5.0,Very Cute!!,1358553600,"01 19, 2013"
2,A1RLQXYNCMWRWN,0000031887,Carola,"[0, 0]",What can I say... my daughters have it in oran...,5.0,I have buy more than one,1357257600,"01 4, 2013"
3,A8U3FAMSJVHS5,0000031887,Caromcg,"[0, 0]","We bought several tutus at once, and they are ...",5.0,"Adorable, Sturdy",1398556800,"04 27, 2014"
4,A3GEOILWLK86XM,0000031887,CJ,"[0, 0]",Thank you Halo Heaven great product for Little...,5.0,Grammy's Angels Love it,1394841600,"03 15, 2014"


In [ ]:
df = df[['reviewText', 'summary', 'overall']]

In [ ]:
df.columns = ['review', 'summary', 'rating']

In [ ]:
df = df.dropna()

In [ ]:
df = df.drop_duplicates()

In [ ]:
def get_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df['sentiment'] = df['rating'].apply(get_sentiment)

In [ ]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df['cleaned_review'] = df['review'].apply(clean_text)

In [ ]:
df = df[df['cleaned_review'].str.len() > 15]

In [ ]:
print(df['sentiment'].value_counts())

sentiment
positive    221048
neutral      30359
negative     26606
Name: count, dtype: int64


In [ ]:
positive_df = df[df['sentiment'] == 'positive'].sample(n=2000, random_state=42)

neutral_df = df[df['sentiment'] == 'neutral'].sample(n=2000, random_state=42)

negative_df = df[df['sentiment'] == 'negative'].sample(n=2000, random_state=42)

In [ ]:
df_balanced = pd.concat([positive_df, neutral_df, negative_df])

In [ ]:
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
print(df_balanced['sentiment'].value_counts())

sentiment
positive    2000
neutral     2000
negative    2000
Name: count, dtype: int64


In [ ]:
df_balanced.to_csv("balanced_reviews_6k.csv", index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_balanced.to_csv('/content/drive/MyDrive/balanced_reviews_6k.csv', index=False)